In [1]:
import os
from google.colab import userdata

# Secretsから取得して環境変数にセット
os.environ['KAGGLE_USERNAME'] = userdata.get('USER')
os.environ['KAGGLE_KEY'] = userdata.get('KEY')

In [2]:
!kaggle competitions download -c home-credit-default-risk

100% 688M/688M [00:04<00:00, 166MB/s]



In [3]:
!unzip /content/home-credit-default-risk.zip

Archive:  /content/home-credit-default-risk.zip
  inflating: HomeCredit_columns_description.csv  
  inflating: POS_CASH_balance.csv    
  inflating: application_test.csv    
  inflating: application_train.csv   
  inflating: bureau.csv              
  inflating: bureau_balance.csv      
  inflating: credit_card_balance.csv  
  inflating: installments_payments.csv  
  inflating: previous_application.csv  
  inflating: sample_submission.csv   


In [4]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score
import warnings
import gc
warnings.filterwarnings('ignore')

# データの読み込み
print("Loading data...")
train = pd.read_csv('application_train.csv')
test = pd.read_csv('application_test.csv')

y_train = train['TARGET']
test_ids = test['SK_ID_CURR']

# 特徴量抽出のために一時的に結合
df = pd.concat([train.drop(columns=['TARGET']), test], axis=0).reset_index(drop=True)
del train, test
gc.collect()
print(f"Initial Dataset shape: {df.shape}")

Loading data...
Initial Dataset shape: (356255, 121)


In [5]:
# 1. ドメイン知識に基づく特徴量設計
df['DAYS_EMPLOYED_ANOM'] = df["DAYS_EMPLOYED"] == 365243
df['DAYS_EMPLOYED'].replace({365243: np.nan}, inplace=True)

df['CREDIT_INCOME_PERCENT'] = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL']
df['ANNUITY_INCOME_PERCENT'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']
df['CREDIT_TERM'] = df['AMT_ANNUITY'] / df['AMT_CREDIT']
df['DAYS_EMPLOYED_PERCENT'] = df['DAYS_EMPLOYED'] / df['DAYS_BIRTH']
df['INCOME_PER_PERSON'] = df['AMT_INCOME_TOTAL'] / df['CNT_FAM_MEMBERS']

# 2. カテゴリカル変数のエンコーディング
cat_cols = [col for col in df.columns if df[col].dtype == 'object']
for col in cat_cols:
    df[col] = df[col].astype('category').cat.codes

# 3. 重要な変数の「二乗項」および「交差項（掛け算・割り算）」の作成
# 特に重要度の高い外部評価スコア(EXT_SOURCE)と雇用・年齢関係に焦点を当てます
high_importance_features = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'DAYS_BIRTH', 'DAYS_EMPLOYED']

print("Generating polynomial and interaction features...")
for col in high_importance_features:
    # 二乗項の作成
    df[f'{col}_SQUARE'] = df[col] ** 2

# 交差項（掛け算・比率）の作成
for i in range(len(high_importance_features)):
    for j in range(i + 1, len(high_importance_features)):
        col1 = high_importance_features[i]
        col2 = high_importance_features[j]
        df[f'{col1}_x_{col2}'] = df[col1] * df[col2]
        # ゼロ除算を防ぎつつ比率を作成
        df[f'{col1}_div_{col2}'] = df[col1] / (df[col2] + 1e-5)

print(f"Dataset shape after feature engineering: {df.shape}")

Generating polynomial and interaction features...
Dataset shape after feature engineering: (356255, 152)


In [6]:
# 訓練データとテストデータに再分割
train_df = df.iloc[:len(y_train)].copy()
test_df = df.iloc[len(y_train):].copy()

features = [col for col in train_df.columns if col not in ['SK_ID_CURR']]

del df
gc.collect()

print(f"Training features count: {len(features)}")

Training features count: 151


In [7]:
import lightgbm as lgb
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score

# 1. 事前に特徴量とターゲットをNumPy配列に変換（Pandasのオーバーヘッドを排除）
X_train_np = train_df[features].values
y_train_np = y_train.values
X_test_np = test_df[features].values

folds = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(train_df))
sub_preds = np.zeros(len(test_df))

# 共通パラメータ（device_type='gpu' のみ指定し、n_jobsは削除・またはCPUスレッドを絞る）
lgb_params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'learning_rate': 0.02,
    'num_leaves': 31,
    'max_depth': 8,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'random_state': 42,
    'verbose': -1,
    'device_type': 'gpu'
}

print("Starting LightGBM GPU cross-validation training (Native API)...")

for fold, (train_idx, val_idx) in enumerate(folds.split(X_train_np, y_train_np)):
    # 2. 訓練データ・検証データをLGBMのDatasetとして作成
    # free_raw_data=Falseにすることでメモリ効率を最適化
    train_data = lgb.Dataset(X_train_np[train_idx], label=y_train_np[train_idx], free_raw_data=False)
    valid_data = lgb.Dataset(X_train_np[val_idx], label=y_train_np[val_idx], reference=train_data, free_raw_data=False)

    # 3. ネイティブAPI (lgb.train) で学習を実行
    model = lgb.train(
        params=lgb_params,
        train_set=train_data,
        num_boost_round=5000,
        valid_sets=[valid_data],
        callbacks=[
            lgb.early_stopping(stopping_rounds=50, verbose=False),
            lgb.log_evaluation(period=100) # 進捗確認用（必要に応じて変更）
        ]
    )

    # 4. 予測の実行
    oof_preds[val_idx] = model.predict(X_train_np[val_idx], num_iteration=model.best_iteration)
    sub_preds += model.predict(X_test_np, num_iteration=model.best_iteration) / folds.n_splits

    score = roc_auc_score(y_train_np[val_idx], oof_preds[val_idx])
    print(f"Fold {fold + 1} AUC: {score:.5f} (Best Iteration: {model.best_iteration})")

overall_auc = roc_auc_score(y_train_np, oof_preds)
print(f"\nOverall Out-of-Fold AUC: {overall_auc:.5f}")

Starting LightGBM GPU cross-validation training (Native API)...
[100]	valid_0's auc: 0.75063
[200]	valid_0's auc: 0.761261
[300]	valid_0's auc: 0.76605
[400]	valid_0's auc: 0.768106
[500]	valid_0's auc: 0.769227
[600]	valid_0's auc: 0.769626
[700]	valid_0's auc: 0.770084
[800]	valid_0's auc: 0.770286
[900]	valid_0's auc: 0.770355
Fold 1 AUC: 0.77045 (Best Iteration: 885)
[100]	valid_0's auc: 0.749115
[200]	valid_0's auc: 0.759861
[300]	valid_0's auc: 0.764425
[400]	valid_0's auc: 0.76609
[500]	valid_0's auc: 0.767338
[600]	valid_0's auc: 0.76814
[700]	valid_0's auc: 0.768865
[800]	valid_0's auc: 0.769045
Fold 2 AUC: 0.76905 (Best Iteration: 752)
[100]	valid_0's auc: 0.746672
[200]	valid_0's auc: 0.757382
[300]	valid_0's auc: 0.761798
[400]	valid_0's auc: 0.764119
[500]	valid_0's auc: 0.764933
[600]	valid_0's auc: 0.765637
[700]	valid_0's auc: 0.765854
Fold 3 AUC: 0.76590 (Best Iteration: 650)
[100]	valid_0's auc: 0.747002
[200]	valid_0's auc: 0.757147
[300]	valid_0's auc: 0.761836
[400

In [8]:
# 提出ファイルの作成
submission = pd.DataFrame({
    'SK_ID_CURR': test_ids,
    'TARGET': sub_preds
})
submission.to_csv('submission_lgb.csv', index=False)
print("Submission file 'submission_lgb.csv' successfully created!")
display(submission.head())

Submission file 'submission_lgb.csv' successfully created!


,SK_ID_CURR,TARGET
0,100001,0.031331
1,100005,0.096952
2,100013,0.015142
3,100028,0.037342
4,100038,0.164908


### すべてのサブテーブルの集約と結合処理
各サブテーブルから統計特徴量（平均、最大、最小、合計など）を抽出し、メインの `train_df` および `test_df` にマージします。メモリ消費を最小限に抑えるため、処理が終わるたびにGC（ガベージコレクション）を実行します。

In [9]:
import pandas as pd
import numpy as np
import gc

# 1. bureau.csv & bureau_balance.csv の処理
print("Processing bureau and bureau_balance...")
bb = pd.read_csv('bureau_balance.csv')
# bureau_balanceのワンホットエンコーディングと集約
bb = pd.get_dummies(bb, columns=['STATUS'], dummy_na=False)
bb_agg = bb.groupby('SK_ID_BUREAU').agg(['mean', 'size'])
bb_agg.columns = pd.Index([e[0] + "_" + e[1].upper() for e in bb_agg.columns.tolist()])
del bb; gc.collect()

bureau = pd.read_csv('bureau.csv')
bureau = bureau.merge(bb_agg, on='SK_ID_BUREAU', how='left')
del bb_agg; gc.collect()

bureau_num_aggregations = {
    'DAYS_CREDIT': ['min', 'max', 'mean', 'var'],
    'DAYS_CREDIT_ENDDATE': ['min', 'max', 'mean'],
    'DAYS_CREDIT_UPDATE': ['mean'],
    'CREDIT_DAY_OVERDUE': ['max', 'mean'],
    'AMT_CREDIT_MAX_OVERDUE': ['mean'],
    'AMT_CREDIT_SUM': ['max', 'mean', 'sum'],
    'AMT_CREDIT_SUM_DEBT': ['max', 'mean', 'sum'],
    'AMT_CREDIT_SUM_LIMIT': ['mean', 'sum'],
    'AMT_CREDIT_SUM_OVERDUE': ['mean'],
    'CNT_CREDIT_PROLONG': ['sum']
}
bureau_agg = bureau.groupby('SK_ID_CURR').agg(bureau_num_aggregations)
bureau_agg.columns = pd.Index(['BUREAU_' + e[0] + "_" + e[1].upper() for e in bureau_agg.columns.tolist()])
bureau_agg['BUREAU_LOAN_COUNT'] = bureau.groupby('SK_ID_CURR').size()

del bureau; gc.collect()
print(f"Bureau features shape: {bureau_agg.shape}")

Processing bureau and bureau_balance...
Bureau features shape: (305811, 22)


In [10]:
# 2. previous_application.csv の処理
print("Processing previous_applications...")
prev = pd.read_csv('previous_application.csv')
prev['DAYS_FIRST_DRAWING'].replace(365243, np.nan, inplace=True)
prev['DAYS_FIRST_DUE'].replace(365243, np.nan, inplace=True)
prev['DAYS_LAST_DUE_1ST_VERSION'].replace(365243, np.nan, inplace=True)
prev['DAYS_LAST_DUE'].replace(365243, np.nan, inplace=True)
prev['DAYS_TERMINATION'].replace(365243, np.nan, inplace=True)

prev_aggregations = {
    'AMT_ANNUITY': ['max', 'mean', 'sum'],
    'AMT_APPLICATION': ['max', 'mean', 'sum'],
    'AMT_CREDIT': ['max', 'mean', 'sum'],
    'AMT_DOWN_PAYMENT': ['max', 'mean'],
    'AMT_GOODS_PRICE': ['max', 'mean', 'sum'],
    'HOUR_APPR_PROCESS_START': ['max', 'mean'],
    'RATE_DOWN_PAYMENT': ['max', 'mean'],
    'DAYS_DECISION': ['min', 'max', 'mean'],
    'CNT_PAYMENT': ['mean', 'sum'],
}
prev_agg = prev.groupby('SK_ID_CURR').agg(prev_aggregations)
prev_agg.columns = pd.Index(['PREV_' + e[0] + "_" + e[1].upper() for e in prev_agg.columns.tolist()])
prev_agg['PREV_APP_COUNT'] = prev.groupby('SK_ID_CURR').size()

del prev; gc.collect()
print(f"Previous App features shape: {prev_agg.shape}")

Processing previous_applications...
Previous App features shape: (338857, 24)


In [11]:
# 3. POS_CASH_balance.csv, installments_payments.csv, credit_card_balance.csv の処理
print("Processing POS_CASH, Installments, Credit Card balances...")

pos = pd.read_csv('POS_CASH_balance.csv')
pos_agg = pos.groupby('SK_ID_CURR').agg({
    'MONTHS_BALANCE': ['max', 'mean', 'size'],
    'CNT_INSTALMENT': ['max', 'mean'],
    'CNT_INSTALMENT_FUTURE': ['max', 'mean'],
    'SK_DPD': ['max', 'mean'],
    'SK_DPD_DEF': ['max', 'mean']
})
pos_agg.columns = pd.Index(['POS_' + e[0] + "_" + e[1].upper() for e in pos_agg.columns.tolist()])
del pos; gc.collect()

inst = pd.read_csv('installments_payments.csv')
inst['DPD'] = inst['DAYS_ENTRY_PAYMENT'] - inst['DAYS_INSTALMENT']
inst['DBD'] = inst['DAYS_INSTALMENT'] - inst['DAYS_ENTRY_PAYMENT']
inst['DPD'] = inst['DPD'].apply(lambda x: x if x > 0 else 0)
inst['DBD'] = inst['DBD'].apply(lambda x: x if x > 0 else 0)
inst_agg = inst.groupby('SK_ID_CURR').agg({
    'NUM_INSTALMENT_VERSION': ['nunique'],
    'DPD': ['max', 'mean', 'sum'],
    'DBD': ['max', 'mean', 'sum'],
    'AMT_INSTALMENT': ['max', 'mean', 'sum'],
    'AMT_PAYMENT': ['min', 'max', 'mean', 'sum'],
})
inst_agg.columns = pd.Index(['INST_' + e[0] + "_" + e[1].upper() for e in inst_agg.columns.tolist()])
del inst; gc.collect()

cc = pd.read_csv('credit_card_balance.csv')
cc_agg = cc.groupby('SK_ID_CURR').agg({
    'MONTHS_BALANCE': ['min', 'max', 'mean'],
    'AMT_BALANCE': ['max', 'mean', 'sum'],
    'AMT_CREDIT_LIMIT_ACTUAL': ['max', 'mean'],
    'AMT_DRAWINGS_ATM_CURRENT': ['max', 'mean', 'sum'],
    'AMT_DRAWINGS_CURRENT': ['max', 'mean', 'sum'],
    'AMT_PAYMENT_TOTAL_CURRENT': ['max', 'mean', 'sum'],
    'CNT_DRAWINGS_ATM_CURRENT': ['max', 'mean'],
    'SK_DPD': ['max', 'mean'],
})
cc_agg.columns = pd.Index(['CC_' + e[0] + "_" + e[1].upper() for e in cc_agg.columns.tolist()])
del cc; gc.collect()

print("Subtables features successfully extracted.")

Processing POS_CASH, Installments, Credit Card balances...
Subtables features successfully extracted.


In [12]:
# 4. すべての特徴量をメインのテーブルにマージ
print("Merging all aggregated tables with Main Dataset...")

train_all_tables = train_df.merge(bureau_agg, on='SK_ID_CURR', how='left')
train_all_tables = train_all_tables.merge(prev_agg, on='SK_ID_CURR', how='left')
train_all_tables = train_all_tables.merge(pos_agg, on='SK_ID_CURR', how='left')
train_all_tables = train_all_tables.merge(inst_agg, on='SK_ID_CURR', how='left')
train_all_tables = train_all_tables.merge(cc_agg, on='SK_ID_CURR', how='left')

test_all_tables = test_df.merge(bureau_agg, on='SK_ID_CURR', how='left')
test_all_tables = test_all_tables.merge(prev_agg, on='SK_ID_CURR', how='left')
test_all_tables = test_all_tables.merge(pos_agg, on='SK_ID_CURR', how='left')
test_all_tables = test_all_tables.merge(inst_agg, on='SK_ID_CURR', how='left')
test_all_tables = test_all_tables.merge(cc_agg, on='SK_ID_CURR', how='left')

# 特徴量リストを抽出（ID除く）
all_features = [col for col in train_all_tables.columns if col not in ['SK_ID_CURR']]

# メモリの最適化
del bureau_agg, prev_agg, pos_agg, inst_agg, cc_agg; gc.collect()
print(f"Final train dataset shape: {train_all_tables.shape}")
print(f"Final test dataset shape: {test_all_tables.shape}")

Merging all aggregated tables with Main Dataset...
Final train dataset shape: (307511, 244)
Final test dataset shape: (48744, 244)


In [14]:
# 5. 全特徴量データセットを用いたGPU高速化クロスバリデーション学習
import lightgbm as lgb
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score

# NumPy配列へ変換しオーバーヘッドを極小化
X_train_all = train_all_tables[all_features].values
y_train_all = y_train_np
X_test_all = test_all_tables[all_features].values

folds = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds_all = np.zeros(len(train_all_tables))
sub_preds_all = np.zeros(len(test_all_tables))

# 拡張特徴量に最適化したGPUハイパーパラメータ設定
lgb_params_all = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'learning_rate': 0.02,
    'num_leaves': 34,
    'max_depth': 8,
    'feature_fraction': 0.7,
    'bagging_fraction': 0.7,
    'bagging_freq': 5,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'random_state': 42,
    'verbose': -1,
    'device_type': 'gpu'
}

print("Starting LightGBM GPU cross-validation training on full enriched dataset...")

for fold, (train_idx, val_idx) in enumerate(folds.split(X_train_all, y_train_all)):
    # Datasetを作成し、メモリ最適化を行う
    train_data = lgb.Dataset(X_train_all[train_idx], label=y_train_all[train_idx], free_raw_data=False)
    valid_data = lgb.Dataset(X_train_all[val_idx], label=y_train_all[val_idx], reference=train_data, free_raw_data=False)

    # 学習
    model = lgb.train(
        params=lgb_params_all,
        train_set=train_data,
        num_boost_round=5000,
        valid_sets=[valid_data],
        callbacks=[
            lgb.early_stopping(stopping_rounds=50, verbose=False),
            lgb.log_evaluation(period=100)
        ]
    )

    # 予測
    oof_preds_all[val_idx] = model.predict(X_train_all[val_idx], num_iteration=model.best_iteration)
    sub_preds_all += model.predict(X_test_all, num_iteration=model.best_iteration) / folds.n_splits

    score = roc_auc_score(y_train_all[val_idx], oof_preds_all[val_idx])
    print(f"Fold {fold + 1} AUC: {score:.5f} (Best Iteration: {model.best_iteration})")

overall_auc_all = roc_auc_score(y_train_all, oof_preds_all)
print(f"\nOverall Out-of-Fold AUC (Full Dataset): {overall_auc_all:.5f}")

Starting LightGBM GPU cross-validation training on full enriched dataset...
[100]	valid_0's auc: 0.759322
[200]	valid_0's auc: 0.773458
[300]	valid_0's auc: 0.779499
[400]	valid_0's auc: 0.782489
[500]	valid_0's auc: 0.784536
[600]	valid_0's auc: 0.78578
[700]	valid_0's auc: 0.786909
[800]	valid_0's auc: 0.787089
Fold 1 AUC: 0.78717 (Best Iteration: 780)
[100]	valid_0's auc: 0.759693
[200]	valid_0's auc: 0.772887
[300]	valid_0's auc: 0.779128
[400]	valid_0's auc: 0.782304
[500]	valid_0's auc: 0.784037
[600]	valid_0's auc: 0.785402
[700]	valid_0's auc: 0.78627
[800]	valid_0's auc: 0.786487
[900]	valid_0's auc: 0.786882
Fold 2 AUC: 0.78698 (Best Iteration: 923)
[100]	valid_0's auc: 0.757768
[200]	valid_0's auc: 0.771705
[300]	valid_0's auc: 0.777545
[400]	valid_0's auc: 0.780414
[500]	valid_0's auc: 0.782234
[600]	valid_0's auc: 0.783687
[700]	valid_0's auc: 0.784263
[800]	valid_0's auc: 0.785031
[900]	valid_0's auc: 0.785461
[1000]	valid_0's auc: 0.78589
[1100]	valid_0's auc: 0.786335
[

In [15]:
# 6. 全テーブル予測ファイルの保存
submission_all = pd.DataFrame({
    'SK_ID_CURR': test_ids,
    'TARGET': sub_preds_all
})
submission_all.to_csv('submission_lgb_all_tables.csv', index=False)
print("Submission file 'submission_lgb_all_tables.csv' successfully created!")
display(submission_all.head())

Submission file 'submission_lgb_all_tables.csv' successfully created!


,SK_ID_CURR,TARGET
0,100001,0.037473
1,100005,0.145733
2,100013,0.022228
3,100028,0.032587
4,100038,0.172517
